# Phone Scrolling Detector



This notebook builds a practical detector for whether a person is likely scrolling on a phone in an image. It does this in two stages:



1. Train one object detector that recognizes `face`, `person`, and `mobile_phone`

2. Convert those detections into a binary `is_scrolling` decision using person/face/phone geometry



## Dataset choices



- **WIDER FACE** for face bounding boxes. In `torchvision`, `WIDERFace(..., download=True)` downloads the official dataset archives and annotations.

- **Open Images V7** for `Person` and `Mobile phone` boxes, loaded as a small targeted subset via FiftyOne.



## Why this design



There is no widely-used official dataset that directly labels the action *scrolling on a phone* as a standalone class. A good practical approximation is:



- detect the face

- detect the phone

- optionally detect the person

- classify the image as *scrolling* when the phone is held in a typical hand-use region relative to the face/body



This is a useful baseline and easy to iterate on once you collect task-specific examples.


## References



- WIDER FACE: https://mmlab.ie.cuhk.edu.hk/projects/WIDERFace/index.html

- Torchvision WIDERFace dataset loader: https://docs.pytorch.org/vision/master/_modules/torchvision/datasets/widerface.html

- Open Images download page: https://storage.googleapis.com/openimages/web/download.html

- FiftyOne Open Images V7 dataset zoo docs: https://docs.voxel51.com/dataset_zoo/datasets/open_images_v7.html



If the WIDER FACE project page is flaky, the `torchvision.datasets.WIDERFace` loader is still a reliable programmatic path because it uses the official archive IDs plus the published annotation zip URL.


In [1]:
%pip install -q ultralytics fiftyone gdown torch torchvision pyyaml tqdm pillow


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations



import math

import os

import random

import shutil

from dataclasses import dataclass

from pathlib import Path

from typing import Iterable



import yaml

from PIL import Image, ImageDraw

from tqdm.auto import tqdm



import torch

from torchvision.datasets import WIDERFace



import fiftyone as fo

import fiftyone.zoo as foz



from ultralytics import YOLO



SEED = 42

random.seed(SEED)

torch.manual_seed(SEED)



@dataclass

class Config:

    project_root: Path = Path.cwd()

    data_root: Path = Path.cwd() / "data"

    raw_root: Path = Path.cwd() / "data" / "raw"

    yolo_root: Path = Path.cwd() / "data" / "scrolling_yolo"

    runs_root: Path = Path.cwd() / "runs"

    notes_root: Path = Path.cwd() / "artifacts"

    open_images_train_samples: int = 1200

    open_images_val_samples: int = 250

    epochs: int = 30

    imgsz: int = 640

    batch: int = 16

    device: str = "0" if torch.cuda.is_available() else "cpu"



cfg = Config()



for path in [cfg.data_root, cfg.raw_root, cfg.yolo_root, cfg.runs_root, cfg.notes_root]:

    path.mkdir(parents=True, exist_ok=True)



CLASS_TO_ID = {"face": 0, "person": 1, "mobile_phone": 2}

ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}



cfg


## Step 1: Download the source datasets



Notes:



- `WIDERFace(..., download=True)` needs `gdown` because the image archives are mirrored via Google Drive in `torchvision`.

- Open Images is large, so we intentionally download only a detection subset containing `Person` and `Mobile phone`.

- Start with the sample counts below, then scale up once the pipeline works locally.


In [ ]:
wider_train = WIDERFace(root=cfg.raw_root, split="train", download=True)

wider_val = WIDERFace(root=cfg.raw_root, split="val", download=True)



oi_train = foz.load_zoo_dataset(

    "open-images-v7",

    split="train",

    label_types=["detections"],

    classes=["Person", "Mobile phone"],

    max_samples=cfg.open_images_train_samples,

    shuffle=True,

    seed=SEED,

    dataset_name="open-images-v7-scroll-train",

)



oi_val = foz.load_zoo_dataset(

    "open-images-v7",

    split="validation",

    label_types=["detections"],

    classes=["Person", "Mobile phone"],

    max_samples=cfg.open_images_val_samples,

    shuffle=True,

    seed=SEED,

    dataset_name="open-images-v7-scroll-val",

)



print(f"WIDER FACE train images: {len(wider_train)}")

print(f"WIDER FACE val images:   {len(wider_val)}")

print(f"Open Images train samples: {len(oi_train)}")

print(f"Open Images val samples:   {len(oi_val)}")


## Step 2: Convert both datasets into one YOLO detection dataset



The combined detector will learn:



- `face` from WIDER FACE

- `person` and `mobile_phone` from Open Images



This cell writes:



- `data/scrolling_yolo/images/train|val`

- `data/scrolling_yolo/labels/train|val`

- `data/scrolling_yolo/scrolling.yaml`


In [ ]:
def reset_split_dirs(root: Path, split: str) -> tuple[Path, Path]:

    images_dir = root / "images" / split

    labels_dir = root / "labels" / split

    if images_dir.exists():

        shutil.rmtree(images_dir)

    if labels_dir.exists():

        shutil.rmtree(labels_dir)

    images_dir.mkdir(parents=True, exist_ok=True)

    labels_dir.mkdir(parents=True, exist_ok=True)

    return images_dir, labels_dir



def xywh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int) -> tuple[float, float, float, float] | None:

    if w <= 1 or h <= 1:

        return None

    x = max(0.0, min(x, img_w - 1))

    y = max(0.0, min(y, img_h - 1))

    w = max(1.0, min(w, img_w - x))

    h = max(1.0, min(h, img_h - y))

    x_center = (x + w / 2.0) / img_w

    y_center = (y + h / 2.0) / img_h

    return x_center, y_center, w / img_w, h / img_h



def save_yolo_label(label_path: Path, rows: list[str]) -> None:

    label_path.write_text("\
".join(rows) + ("\
" if rows else ""), encoding="utf-8")



def widerface_to_yolo(dataset: WIDERFace, split: str, images_dir: Path, labels_dir: Path, limit: int | None = None) -> int:

    written = 0

    iterable = dataset.img_info if limit is None else dataset.img_info[:limit]

    for idx, info in enumerate(tqdm(iterable, desc=f"WIDERFace -> {split}")):

        image_path = Path(info["img_path"])

        with Image.open(image_path) as im:

            img_w, img_h = im.size

        boxes = info["annotations"]["bbox"].tolist()

        rows = []

        for x, y, w, h in boxes:

            converted = xywh_to_yolo(float(x), float(y), float(w), float(h), img_w, img_h)

            if converted is None:

                continue

            xc, yc, wn, hn = converted

            rows.append(f"{CLASS_TO_ID['face']} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

        if not rows:

            continue

        target_name = f"wider_{split}_{idx:06d}{image_path.suffix.lower()}"

        shutil.copy2(image_path, images_dir / target_name)

        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)

        written += 1

    return written



def find_fiftyone_detection_field(dataset: fo.Dataset) -> str:

    for field_name, field in dataset.get_field_schema().items():

        document_type = getattr(field, "document_type", None)

        if document_type is not None and document_type.__name__ == "Detections":

            return field_name

    raise RuntimeError("No FiftyOne Detections field found in dataset schema")



def openimages_to_yolo(dataset: fo.Dataset, split: str, images_dir: Path, labels_dir: Path) -> int:

    det_field = find_fiftyone_detection_field(dataset)

    class_map = {"Person": CLASS_TO_ID["person"], "Mobile phone": CLASS_TO_ID["mobile_phone"]}

    written = 0

    for idx, sample in enumerate(tqdm(dataset.iter_samples(progress=True, autosave=False), desc=f"OpenImages -> {split}")):

        detections = getattr(sample, det_field, None)

        if detections is None or not detections.detections:

            continue

        image_path = Path(sample.filepath)

        rows = []

        for det in detections.detections:

            if det.label not in class_map:

                continue

            x, y, w, h = det.bounding_box

            xc = x + (w / 2.0)

            yc = y + (h / 2.0)

            rows.append(f"{class_map[det.label]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        if not rows:

            continue

        target_name = f"oi_{split}_{idx:06d}{image_path.suffix.lower()}"

        shutil.copy2(image_path, images_dir / target_name)

        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)

        written += 1

    return written



train_images, train_labels = reset_split_dirs(cfg.yolo_root, "train")

val_images, val_labels = reset_split_dirs(cfg.yolo_root, "val")



wider_train_written = widerface_to_yolo(wider_train, "train", train_images, train_labels)

wider_val_written = widerface_to_yolo(wider_val, "val", val_images, val_labels)

oi_train_written = openimages_to_yolo(oi_train, "train", train_images, train_labels)

oi_val_written = openimages_to_yolo(oi_val, "val", val_images, val_labels)



yaml_path = cfg.yolo_root / "scrolling.yaml"

yaml_payload = {

    "path": str(cfg.yolo_root.resolve()),

    "train": "images/train",

    "val": "images/val",

    "names": ["face", "person", "mobile_phone"],

}

yaml_path.write_text(yaml.safe_dump(yaml_payload, sort_keys=False), encoding="utf-8")



print({

    "wider_train_written": wider_train_written,

    "wider_val_written": wider_val_written,

    "oi_train_written": oi_train_written,

    "oi_val_written": oi_val_written,

    "yaml": str(yaml_path),

})


## Step 3: Train the detector



A small YOLO model is a good baseline here. Swap `yolo11n.pt` for a larger checkpoint if you have more compute.


In [ ]:
detector = YOLO("yolo11n.pt")



train_results = detector.train(

    data=str(yaml_path),

    epochs=cfg.epochs,

    imgsz=cfg.imgsz,

    batch=cfg.batch,

    device=cfg.device,

    project=str(cfg.runs_root),

    name="phone_scrolling_detector",

    exist_ok=True,

    pretrained=True,

    workers=2,

)



best_weights = Path(detector.trainer.best)

best_weights


## Step 4: Convert detections into a binary `is_scrolling` signal



Heuristic used here:



- there is at least one detected face

- there is at least one detected phone

- ideally both belong to the same person box

- the phone is in front of the torso and below the face center

- the phone is not too far horizontally from the face



This is intentionally simple. Once you collect labeled positive and negative examples, you can replace this rule block with a learned classifier.


In [ ]:
trained_detector = YOLO(str(best_weights))



def xyxy_center(box: list[float]) -> tuple[float, float]:

    x1, y1, x2, y2 = box

    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)



def box_size(box: list[float]) -> tuple[float, float]:

    x1, y1, x2, y2 = box

    return (x2 - x1, y2 - y1)



def contains_point(box: list[float], point: tuple[float, float]) -> bool:

    x1, y1, x2, y2 = box

    px, py = point

    return x1 <= px <= x2 and y1 <= py <= y2



def to_named_detections(result, conf: float = 0.25) -> dict[str, list[dict]]:

    output = {"face": [], "person": [], "mobile_phone": []}

    names = result.names

    boxes = result.boxes

    for cls_id, score, xyxy in zip(boxes.cls.tolist(), boxes.conf.tolist(), boxes.xyxy.tolist()):

        if score < conf:

            continue

        label = names[int(cls_id)]

        output[label].append({"box": xyxy, "score": float(score)})

    return output



def scrolling_score(named: dict[str, list[dict]]) -> tuple[float, dict]:

    faces = named["face"]

    phones = named["mobile_phone"]

    persons = named["person"]

    if not faces or not phones:

        return 0.0, {"reason": "missing_face_or_phone"}



    best_score = 0.0

    best_meta = {"reason": "no_match"}



    for face in faces:

        face_box = face["box"]

        face_cx, face_cy = xyxy_center(face_box)

        face_w, face_h = box_size(face_box)



        matching_people = [p for p in persons if contains_point(p["box"], (face_cx, face_cy))]

        candidate_people = matching_people or persons or [{"box": [face_box[0] - 2.0 * face_w, face_box[1], face_box[2] + 2.0 * face_w, face_box[3] + 6.0 * face_h]}]



        for person in candidate_people:

            person_box = person["box"]

            person_w, person_h = box_size(person_box)

            if person_w <= 1 or person_h <= 1:

                continue

            for phone in phones:

                phone_box = phone["box"]

                phone_cx, phone_cy = xyxy_center(phone_box)

                phone_w, phone_h = box_size(phone_box)



                if not contains_point(person_box, (phone_cx, phone_cy)):

                    continue



                horizontal_distance = abs(phone_cx - face_cx) / max(person_w, 1.0)

                vertical_offset = (phone_cy - face_cy) / max(person_h, 1.0)

                phone_scale = max(phone_w, phone_h) / max(person_w, 1.0)



                horizontal_ok = horizontal_distance <= 0.35

                vertical_ok = 0.02 <= vertical_offset <= 0.45

                scale_ok = 0.03 <= phone_scale <= 0.30



                score = 0.0

                score += 0.35 if horizontal_ok else max(0.0, 0.35 - horizontal_distance)

                score += 0.35 if vertical_ok else max(0.0, 0.35 - abs(vertical_offset - 0.18))

                score += 0.20 if scale_ok else max(0.0, 0.20 - abs(phone_scale - 0.10))

                score += 0.10 * min(face["score"], phone["score"])



                if score > best_score:

                    best_score = float(min(score, 1.0))

                    best_meta = {

                        "face_box": face_box,

                        "phone_box": phone_box,

                        "person_box": person_box,

                        "horizontal_distance": horizontal_distance,

                        "vertical_offset": vertical_offset,

                        "phone_scale": phone_scale,

                    }

    return best_score, best_meta



def predict_scrolling(image_path: str | Path, det_conf: float = 0.25, scroll_threshold: float = 0.60):

    result = trained_detector.predict(source=str(image_path), conf=det_conf, verbose=False)[0]

    named = to_named_detections(result, conf=det_conf)

    score, meta = scrolling_score(named)

    return {

        "image_path": str(image_path),

        "is_scrolling": score >= scroll_threshold,

        "scroll_score": round(score, 4),

        "detections": named,

        "meta": meta,

    }


## Step 5: Try it on an image



Put a test image path in `TEST_IMAGE`. If you have a webcam or short clips, you can run this function frame-by-frame.


In [ ]:
TEST_IMAGE = None



if TEST_IMAGE is not None:

    prediction = predict_scrolling(TEST_IMAGE)

    print(prediction["is_scrolling"], prediction["scroll_score"])

    print(prediction["meta"])

else:

    print("Set TEST_IMAGE to a local image path before running this cell.")


In [ ]:
def draw_prediction(image_path: str | Path, prediction: dict) -> Image.Image:

    im = Image.open(image_path).convert("RGB")

    draw = ImageDraw.Draw(im)

    color_map = {"face": "lime", "person": "cyan", "mobile_phone": "yellow"}

    for label, detections in prediction["detections"].items():

        for det in detections:

            x1, y1, x2, y2 = det["box"]

            draw.rectangle([x1, y1, x2, y2], outline=color_map[label], width=3)

            draw.text((x1 + 4, max(0, y1 - 14)), f"{label}:{det['score']:.2f}", fill=color_map[label])

    draw.text((12, 12), f"scrolling={prediction['is_scrolling']} score={prediction['scroll_score']:.2f}", fill="white")

    return im



if TEST_IMAGE is not None:

    draw_prediction(TEST_IMAGE, prediction)


## Next improvements



Strong directions if you want better accuracy:



- collect a small labeled dataset with true `scrolling` / `not_scrolling` examples from your target camera angle

- add a face landmark or gaze model so you can verify the user is looking downward toward the phone

- run the detector on video and require the phone to remain present for several consecutive frames

- replace the hand-written rule with a shallow learned classifier trained on detection geometry features
